In [0]:
%skip
import time
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# ── SECTION 1: SPARK ──────────────────────────────────────────────────────────
try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless(True).getOrCreate()

# ── SECTION 2: REGISTRY ───────────────────────────────────────────────────────
BUCKET  = "s3://project-racing-bronze"
CATALOG = "harness_stream.bronze"

STREAMS = {
    "betfair_catalogue": {
        "source_path":     f"{BUCKET}/betfair/market_catalogue/",
        "schema_location": f"{BUCKET}/_schema/betfair_catalogue/",
        "checkpoint":      f"{BUCKET}/_checkpoints/betfair_catalogue/",
        "target_table":    f"{CATALOG}.betfair_catalogue",
        "merge_keys":      ["market_id"],
        "schema_hints":    "extracted_date STRING, race_code STRING, run_time STRING, snapshot_type STRING, runners STRING, event STRING, eventType STRING",
        "partition_col":   "extracted_date",
        "max_files":       10000,
    },
    "betfair_market_book": {
        "source_path":     f"{BUCKET}/betfair/market_book/",
        "schema_location": f"{BUCKET}/_schema/betfair_market_book/",
        "checkpoint":      f"{BUCKET}/_checkpoints/betfair_market_book/",
        "target_table":    f"{CATALOG}.betfair_market_book",
        "merge_keys":      ["marketId", "snapshot_type", "run_time"],
        "schema_hints":    "extracted_date STRING, race_code STRING, run_time STRING, snapshot_type STRING, runners STRING, betDelayModels STRING",
        "partition_col":   "extracted_date",
        "max_files":       20000,
    },
    "formfav_meetings": {
        "source_path":     f"{BUCKET}/formfav/meetings/",
        "schema_location": f"{BUCKET}/_schema/formfav_meetings/",
        "checkpoint":      f"{BUCKET}/_checkpoints/formfav_meetings/",
        "target_table":    f"{CATALOG}.harness_meetings",
        "merge_keys":      ["track_slug", "meeting_date", "race_code"],
        "schema_hints":    (
            "track_slug STRING, meeting_date STRING, race_code STRING, "
            "name STRING, country STRING, country_name STRING, state STRING, "
            "number_of_races INT, track_condition STRING, weather STRING, "
            "_ingested_at STRING, _source_system STRING, _pipeline_version STRING"
        ),
        "partition_col":   "meeting_date",
        "derive_extracted_date": True,
        "max_files":       10000,
    },
    "formfav_races": {
        "source_path":     f"{BUCKET}/formfav/races/",
        "schema_location": f"{BUCKET}/_schema/formfav_races_bulk/",
        "checkpoint":      f"{BUCKET}/_checkpoints/formfav_races_bulk/",
        "target_table":    f"{CATALOG}.harness_races",
        "merge_keys":      ["track_slug", "race_date", "race_number", "race_code"],
        "schema_hints":    (
            "track_slug STRING, race_date STRING, race_number INT, race_code STRING, "
            "race_name STRING, distance_raw STRING, condition STRING, race_class STRING, "
            "prize_money_raw STRING, number_of_runners INT, country STRING, "
            "country_name STRING, state STRING, "
            "_ingested_at STRING, _source_system STRING, _pipeline_version STRING"
        ),
        "partition_col":   "race_date",
        "derive_extracted_date": True,
        "max_files":       10000,
    },
}

# ── SECTION 3: HELPERS ────────────────────────────────────────────────────────
def build_stream(cfg):
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", cfg["schema_location"])
        .option("cloudFiles.schemaHints", cfg["schema_hints"])
        .option("cloudFiles.maxFilesPerTrigger", cfg["max_files"])
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("recursiveFileLookup", "true")
        .load(cfg["source_path"])
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .withColumn("_bronze_loaded_at", F.from_utc_timestamp(F.current_timestamp(), "Australia/Sydney"))
    )

def make_writer(cfg):
    merge_keys    = cfg["merge_keys"]
    target_table  = cfg["target_table"]
    partition_col = cfg["partition_col"]
    derive_date   = cfg.get("derive_extracted_date", False)

    def write_batch(batch_df, batch_id):
        if batch_df.isEmpty():
            return
        df = batch_df
        if derive_date:
            df = df.withColumn("extracted_date", F.col(partition_col))
        deduped    = df.dropDuplicates(merge_keys)
        condition  = " AND ".join(f"t.{k} = s.{k}" for k in merge_keys)
        try:
            if spark.catalog.tableExists(target_table):
                DeltaTable.forName(spark, target_table) \
                    .alias("t").merge(deduped.alias("s"), condition) \
                    .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
            else:
                deduped.write.format("delta").mode("overwrite") \
                    .partitionBy(partition_col).saveAsTable(target_table)
        except Exception as e:
            print(f"❌ Batch {batch_id} error on {target_table}: {e}")
            raise

    return write_batch

# ── SECTION 4: RUNNER ─────────────────────────────────────────────────────────
def run_bulk(streams=None):
    """
    Load all pending files from S3 into Delta.
    streams: list of stream names to run, e.g. ["formfav_meetings", "formfav_races"]
             None = run all streams.
    """
    targets = {k: v for k, v in STREAMS.items() if streams is None or k in streams}

    for name, cfg in targets.items():
        start = time.time()
        print(f"\n🚀 {name} — starting bulk load")
        try:
            (
                build_stream(cfg)
                .writeStream
                .foreachBatch(make_writer(cfg))
                .option("checkpointLocation", cfg["checkpoint"])
                .trigger(availableNow=True)
                .start()
                .awaitTermination()
            )
            count    = spark.table(cfg["target_table"]).count()
            duration = (time.time() - start) / 60
            print(f"✅ {name} done — {count:,} rows | {duration:.1f} min")
        except Exception as e:
            print(f"💥 {name} failed: {e}")
            raise


if __name__ == "__main__":
    # Run only formfav streams
    run_bulk(streams=["formfav_meetings", "formfav_races"])

    # To run everything:
    # run_bulk()


In [0]:
%skip
%sql
ALTER TABLE harness_stream.bronze.betfair_catalogue 
SET TBLPROPERTIES (
  delta.autoOptimize.optimizeWrite = true,
  delta.autoOptimize.autoCompact = true
);

ALTER TABLE harness_stream.bronze.betfair_market_book 
SET TBLPROPERTIES (
  delta.autoOptimize.optimizeWrite = true,
  delta.autoOptimize.autoCompact = true
);

In [0]:
%skip
%sql
OPTIMIZE harness_stream.bronze.betfair_catalogue ZORDER BY (_bronze_loaded_at);
OPTIMIZE harness_stream.bronze.betfair_market_book ZORDER BY (_bronze_loaded_at);

In [0]:
from pyspark.sql import functions as F

races = (
    spark.read
    .option("recursiveFileLookup", "true")
    .json("s3://project-racing-bronze/formfav/races/")
    .withColumn("extracted_date", F.col("race_date"))
    .withColumn("_source_file", F.lit("backfill"))
    .withColumn("_bronze_loaded_at", F.from_utc_timestamp(F.current_timestamp(), "Australia/Sydney"))
)

races.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .partitionBy("race_date").saveAsTable("harness_stream.bronze.harness_races")

print(f"✅ Races — {spark.table('harness_stream.bronze.harness_races').count():,} rows loaded.")


In [0]:
missing_ids = spark.sql("""
    SELECT DISTINCT c.market_id
    FROM harness_stream.bronze.betfair_catalogue c
    LEFT JOIN harness_stream.bronze.betfair_market_book b ON c.market_id = b.marketId
    WHERE b.marketId IS NULL
""")
missing_ids.display()

In [0]:
from pyspark.sql import functions as F

id_list = [r.market_id for r in missing_ids.collect() if r.market_id is not None]

found = (
    spark.read
    .option("recursiveFileLookup", "true")
    .json("s3://project-racing-bronze/betfair/market_book/")
    .filter(F.col("marketId").isin(id_list))
)

print(f"Records found in S3: {found.count()}")

In [0]:
from pyspark.sql import functions as F

found_fixed = found \
    .withColumn("race_code",       F.lit(None).cast("string")) \
    .withColumn("extracted_date",  F.lit("2026-05-15")) \
    .withColumn("_rescued_data",   F.lit(None).cast("string")) \
    .withColumn("_source_file",    F.lit("manual_backfill")) \
    .withColumn("_bronze_loaded_at", F.from_utc_timestamp(F.current_timestamp(), "Australia/Sydney")) \
    .withColumn("betDelayModels",  F.to_json(F.col("betDelayModels"))) \
    .withColumn("runners",         F.to_json(F.col("runners")))

found_fixed.write.format("delta").mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("harness_stream.bronze.betfair_market_book")

still_missing = spark.sql("""
    SELECT COUNT(DISTINCT c.market_id)
    FROM harness_stream.bronze.betfair_catalogue c
    LEFT JOIN harness_stream.bronze.betfair_market_book b ON c.market_id = b.marketId
    WHERE b.marketId IS NULL
""").collect()[0][0]
print(f"✅ Done. Still missing: {still_missing}")

In [0]:
still_missing_ids = spark.sql("""
    SELECT DISTINCT c.market_id
    FROM harness_stream.bronze.betfair_catalogue c
    LEFT JOIN harness_stream.bronze.betfair_market_book b ON c.market_id = b.marketId
    WHERE b.marketId IS NULL AND c.market_id IS NOT NULL
""")

id_list2 = [r.market_id for r in still_missing_ids.collect()]

found2 = (
    spark.read
    .option("recursiveFileLookup", "true")
    .json("s3://project-racing-bronze/betfair/market_book/")
    .filter(F.col("marketId").isin(id_list2))
)

print(f"Records in S3: {found2.count()}")